# Student Health Risk — E002 Tuned

Public LB **0.94960** üreten akış: V2-Core features + 3-fold LightGBM + class multipliers.

## 1. Setup

In [ ]:
from pathlib import Path
import warnings

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import balanced_accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

SEED = 42
ID_COL = "id"
TARGET = "health_condition"
CLASSES = ["at-risk", "fit", "unhealthy"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASSES)}
NUM_COLS = [
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]
CAT_COLS = [
    "diet_type", "stress_level", "sleep_quality", "physical_activity_level",
    "smoking_alcohol", "gender",
]
RATIO_DEFS = {
    "calorie_per_step": ("calorie_expenditure", "step_count"),
    "calorie_per_exercise_min": ("calorie_expenditure", "exercise_duration"),
    "step_per_exercise_min": ("step_count", "exercise_duration"),
    "water_per_bmi": ("water_intake", "bmi"),
    "exercise_per_bmi": ("exercise_duration", "bmi"),
    "steps_per_sleep_hour": ("step_count", "sleep_duration"),
}
RATIO_COLS = list(RATIO_DEFS)
INTERACTION_DEFS = {
    "stress_sleep_quality": ("stress_level", "sleep_quality"),
    "activity_diet": ("physical_activity_level", "diet_type"),
    "smoking_activity": ("smoking_alcohol", "physical_activity_level"),
}

## 2. Data Loading

In [ ]:
def locate_data():
    roots = [Path("/kaggle/input"), Path("../input"), Path("data/playground-series-s6e7"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        train_files = list(root.rglob("train.csv"))
        for train_path in train_files:
            test_path = train_path.with_name("test.csv")
            sample_path = train_path.with_name("sample_submission.csv")
            if test_path.exists() and sample_path.exists():
                return train_path, test_path, sample_path
    raise FileNotFoundError("train.csv, test.csv and sample_submission.csv were not found")

TRAIN_PATH, TEST_PATH, SAMPLE_PATH = locate_data()
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

print("Data directory:", TRAIN_PATH.parent)
print("Train/Test:", train.shape, test.shape)
display(train.head())

In [ ]:
assert TARGET in train and TARGET not in test
assert train[ID_COL].is_unique and test[ID_COL].is_unique
assert set(train[TARGET].unique()) == set(CLASSES)
assert list(sample_submission.columns) == [ID_COL, TARGET]

schema = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing_train": train.isna().sum(),
    "missing_test": test.isna().sum().reindex(train.columns),
    "nunique": train.nunique(dropna=False),
})
display(schema)
print("Duplicate train rows:", train.duplicated().sum())

## 3. Exploratory Data Analysis

### 3.1 Target and Missing Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
order = train[TARGET].value_counts().index
sns.countplot(data=train, x=TARGET, order=order, ax=axes[0], palette="viridis")
axes[0].set_title("Target distribution")
axes[0].tick_params(axis="x", rotation=15)

missing = pd.DataFrame({"train": train.isna().mean(), "test": test.isna().mean()}).fillna(0)
missing = missing.loc[missing.max(axis=1).gt(0)].sort_values("train", ascending=False)
missing.plot(kind="bar", ax=axes[1], color=["#4C72B0", "#DD8452"])
axes[1].set_title("Missing-value rate")
axes[1].set_ylabel("rate")
plt.tight_layout()
plt.show()

display((train[TARGET].value_counts(normalize=True).mul(100).rename("percent").to_frame()).round(3))

### 3.2 Numerical Distributions

In [ ]:
plot_train = train.sample(min(60_000, len(train)), random_state=SEED)
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
for ax, col in zip(axes.flat, NUM_COLS):
    sns.histplot(data=plot_train, x=col, hue=TARGET, stat="density", common_norm=False,
                 element="step", fill=False, bins=35, ax=ax)
    ax.set_title(col)
for ax in axes.flat[len(NUM_COLS):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3.3 Categorical Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, col in zip(axes.flat, CAT_COLS):
    rates = pd.crosstab(train[col].fillna("missing"), train[TARGET], normalize="index")
    rates[CLASSES].plot(kind="bar", stacked=True, ax=ax, colormap="viridis", legend=False)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## 4. Outlier Analysis

In [ ]:
quantiles = train[NUM_COLS].quantile([0.005, 0.25, 0.5, 0.75, 0.995]).T
quantiles["outside_0.5_99.5_pct"] = [
    ((train[c] < quantiles.loc[c, 0.005]) | (train[c] > quantiles.loc[c, 0.995])).mean() * 100
    for c in NUM_COLS
]
display(quantiles.round(3))

fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for ax, col in zip(axes.flat, NUM_COLS):
    sns.boxplot(data=plot_train, x=TARGET, y=col, order=CLASSES, showfliers=False, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=20)
for ax in axes.flat[len(NUM_COLS):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Fold-Safe Feature Engineering

In [ ]:
class V2CorePreprocessor:
    def fit(self, frame):
        self.medians_ = frame[NUM_COLS].median()
        prepared = self._base(frame)
        outlier_sources = NUM_COLS + RATIO_COLS
        self.bounds_ = {
            col: tuple(prepared[col].quantile([0.005, 0.995])) for col in outlier_sources
        }
        flagged = self._flags(prepared)
        candidates = [f"{col}_outlier_{side}" for col in outlier_sources for side in ("low", "high")]
        self.active_flags_ = [c for c in candidates if flagged[c].nunique(dropna=False) > 1]

        categorical = CAT_COLS + list(INTERACTION_DEFS)
        self.category_levels_ = {
            col: sorted(flagged[col].astype(str).unique().tolist()) + ["__UNKNOWN__"]
            for col in categorical
        }
        transformed = self._finish(flagged)
        self.feature_names_ = transformed.columns.tolist()
        return self

    def transform(self, frame):
        flagged = self._flags(self._base(frame))
        return self._finish(flagged).reindex(columns=self.feature_names_)

    def fit_transform(self, frame):
        return self.fit(frame).transform(frame)

    def _base(self, frame):
        result = frame[NUM_COLS + CAT_COLS].copy()
        raw_missing = result.isna()
        for col in NUM_COLS + CAT_COLS:
            result[f"{col}_is_missing"] = raw_missing[col].astype("int8")
        result["missing_count"] = raw_missing.sum(axis=1).astype("int16")
        result[NUM_COLS] = result[NUM_COLS].fillna(self.medians_)
        result[CAT_COLS] = result[CAT_COLS].fillna("missing")

        for output, (numerator, denominator) in RATIO_DEFS.items():
            result[output] = result[numerator].div(result[denominator].add(1)).replace([np.inf, -np.inf], np.nan)
        for output, (left, right) in INTERACTION_DEFS.items():
            result[output] = result[left].astype(str) + "__" + result[right].astype(str)
        return result

    def _flags(self, result):
        result = result.copy()
        for col, (low, high) in self.bounds_.items():
            result[f"{col}_outlier_low"] = (result[col] < low).astype("int8")
            result[f"{col}_outlier_high"] = (result[col] > high).astype("int8")
        return result

    def _finish(self, result):
        result = result.copy()
        result["outlier_count"] = result[self.active_flags_].sum(axis=1).astype("int16")
        keep = (NUM_COLS + CAT_COLS +
                [f"{c}_is_missing" for c in NUM_COLS + CAT_COLS] +
                ["missing_count"] + RATIO_COLS + list(INTERACTION_DEFS) +
                self.active_flags_ + ["outlier_count"])
        result = result[keep]
        for col, levels in self.category_levels_.items():
            values = result[col].astype(str)
            result[col] = pd.Categorical(values.where(values.isin(levels), "__UNKNOWN__"), categories=levels)
        return result

In [ ]:
# Preview only; training below fits medians, bounds and categories separately inside each fold.
preview_processor = V2CorePreprocessor()
preview_features = preview_processor.fit_transform(train.drop(columns=[ID_COL, TARGET]).head(100_000))
print("Feature count:", preview_features.shape[1])
display(preview_features.head())
del preview_features, preview_processor

## 6. Model Training and Validation

In [ ]:
MODEL_PARAMS = dict(
    objective="multiclass", num_class=3, learning_rate=0.035, n_estimators=12_000,
    num_leaves=96, max_depth=-1, min_child_samples=200, subsample=0.85,
    subsample_freq=1, colsample_bytree=0.90, reg_alpha=0.1, reg_lambda=2.0,
    max_bin=255, random_state=SEED, bagging_seed=SEED, feature_fraction_seed=SEED,
    data_random_seed=SEED, deterministic=True, force_col_wise=True, n_jobs=-1, verbosity=-1,
)

X = train.drop(columns=[ID_COL, TARGET])
X_test = test.drop(columns=ID_COL)
y = train[TARGET].map(CLASS_TO_ID).to_numpy()

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
oof_proba = np.zeros((len(train), len(CLASSES)), dtype=np.float32)
test_proba = np.zeros((len(test), len(CLASSES)), dtype=np.float32)
fold_scores, fold_importance = [], []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), 1):
    processor = V2CorePreprocessor()
    X_tr = processor.fit_transform(X.iloc[train_idx])
    X_va = processor.transform(X.iloc[valid_idx])
    X_te = processor.transform(X_test)

    model = lgb.LGBMClassifier(**MODEL_PARAMS)
    model.fit(
        X_tr, y[train_idx],
        eval_set=[(X_va, y[valid_idx])],
        eval_metric="multi_logloss",
        categorical_feature="auto",
        callbacks=[lgb.early_stopping(300, first_metric_only=True, verbose=False)],
    )
    valid_proba = model.predict_proba(X_va, num_iteration=model.best_iteration_)
    fold_test_proba = model.predict_proba(X_te, num_iteration=model.best_iteration_)
    valid_proba /= valid_proba.sum(axis=1, keepdims=True)
    fold_test_proba /= fold_test_proba.sum(axis=1, keepdims=True)
    oof_proba[valid_idx] = valid_proba
    test_proba += fold_test_proba.astype(np.float32) / cv.n_splits

    score = balanced_accuracy_score(y[valid_idx], valid_proba.argmax(1))
    fold_scores.append(score)
    fold_importance.append(pd.DataFrame({
        "feature": X_tr.columns,
        "gain": model.booster_.feature_importance("gain"),
        "fold": fold,
    }))
    print(f"Fold {fold}: balanced accuracy={score:.6f}, best_iteration={model.best_iteration_}")
    del X_tr, X_va, X_te, model

print(f"Raw CV: {np.mean(fold_scores):.6f} ± {np.std(fold_scores, ddof=1):.6f}")

## 7. Evaluation

In [ ]:
raw_pred = oof_proba.argmax(axis=1)
print("Raw OOF balanced accuracy:", balanced_accuracy_score(y, raw_pred))
print(classification_report(y, raw_pred, target_names=CLASSES, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y, raw_pred, display_labels=CLASSES, normalize="true", cmap="Blues", ax=axes[0])
axes[0].set_title("Normalized OOF confusion matrix")

importance = (pd.concat(fold_importance).groupby("feature", as_index=False)["gain"].mean()
              .sort_values("gain", ascending=False).head(20))
sns.barplot(data=importance, y="feature", x="gain", ax=axes[1], color="#4C72B0")
axes[1].set_title("Top-20 mean gain importance")
plt.tight_layout()
plt.show()

## 8. Validated Class-Multiplier Post-processing

In [ ]:
# OOF-selected and public-LB validated E002 multipliers.
MULTIPLIERS = np.array([0.18922984217739805, 1.444453232368245, 1.3663169254543568])
tuned_pred = np.argmax(oof_proba * MULTIPLIERS, axis=1)
tuned_score = balanced_accuracy_score(y, tuned_pred)

print("Tuned OOF balanced accuracy:", round(tuned_score, 6))
print(classification_report(y, tuned_pred, target_names=CLASSES, digits=4))
display(pd.Series(np.array(CLASSES)[tuned_pred]).value_counts(normalize=True).rename("OOF prediction rate"))

## 9. Create `submission.csv`

In [ ]:
test_pred = np.argmax(test_proba * MULTIPLIERS, axis=1)
submission = sample_submission[[ID_COL]].copy()
submission[TARGET] = np.array(CLASSES)[test_pred]

assert submission.shape == sample_submission.shape
assert submission[ID_COL].equals(sample_submission[ID_COL])
assert submission[TARGET].isin(CLASSES).all()
assert submission[TARGET].notna().all()

submission.to_csv("submission.csv", index=False)
print("Saved: submission.csv", submission.shape)
display(submission.head())
display(submission[TARGET].value_counts(normalize=True).rename("test prediction rate"))